In [ ]:
!mkdir -p data/narcbench/core data/narcbench/transfer

BASE = "https://huggingface.co/datasets/aaronrose227/narcbench/resolve/main/activations/qwen3_32b"
!wget -q -O data/narcbench/core/activations.npz {BASE}/core/activations_gen.npz
!wget -q -O data/narcbench/core/metadata.json {BASE}/core/metadata_gen.json
!wget -q -O data/narcbench/transfer/activations.npz {BASE}/transfer/activations_gen.npz
!wget -q -O data/narcbench/transfer/metadata.json {BASE}/transfer/metadata_gen.json
!ls -la data/narcbench/*/


In [ ]:
!mkdir -p data/narcbench/stego
BASE = "https://huggingface.co/datasets/aaronrose227/narcbench/resolve/main/activations/qwen3_32b"
!wget -q -O data/narcbench/stego/activations.npz {BASE}/stego/activations_gen.npz
!wget -q -O data/narcbench/stego/metadata.json {BASE}/stego/metadata_gen.json

In [ ]:
!git clone https://github.com/aaronrose227/narcbench.git
%cd narcbench
#clone narcbench

In [ ]:
!pip install -r requirements.txt
#install requirements

In [ ]:
!mkdir -p data/activations/qwen3_32b/transfer
!wget -O data/activations/qwen3_32b/transfer/activations_gen.npz \
    https://huggingface.co/datasets/aaronrose227/narcbench/resolve/main/activations/qwen3_32b/transfer/activations_gen.npz
!wget -O data/activations/qwen3_32b/transfer/metadata_gen.json \
    https://huggingface.co/datasets/aaronrose227/narcbench/resolve/main/activations/qwen3_32b/transfer/metadata_gen.json

In [ ]:
!mkdir -p data/activations/qwen3_32b/stego
!wget -O data/activations/qwen3_32b/stego/activations_gen.npz \
    https://huggingface.co/datasets/aaronrose227/narcbench/resolve/main/activations/qwen3_32b/stego/activations_gen.npz
!wget -O data/activations/qwen3_32b/stego/metadata_gen.json \
    https://huggingface.co/datasets/aaronrose227/narcbench/resolve/main/activations/qwen3_32b/stego/metadata_gen.json

In [ ]:
!PYTHONPATH=. python3 probes/reproduce.py --model Qwen/Qwen3-32B-AWQ
#reproduce the results


In [ ]:
# ========== INSTALLS ==========
!pip install -q git+https://github.com/anthropics/jacobian-lens
!pip install -q huggingface_hub safetensors scikit-learn

import numpy as np
import torch
import torch.nn.functional as F
import json
from huggingface_hub import hf_hub_download
from safetensors import safe_open
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.metrics import roc_auc_score
from transformers import AutoTokenizer

# ========== LOAD LENS ==========
qwen32b_path = hf_hub_download(
    "neuronpedia/jacobian-lens",
    "qwen3-32b/jlens/Salesforce-wikitext/Qwen3-32B_jacobian_lens.pt"
)
ckpt = torch.load(qwen32b_path, map_location="cpu", weights_only=True)
target_layers = [26, 27, 28, 29, 30]
jacobians = {layer: ckpt['jacobian_sum'][layer] / ckpt['n_done'] for layer in target_layers}
del ckpt

# ========== LOAD UNEMBEDDING ==========
shard_path = hf_hub_download("Qwen/Qwen3-32B-AWQ", "model-00004-of-00004.safetensors")
with safe_open(shard_path, framework="pt", device="cpu") as f:
    W_U = f.get_tensor("lm_head.weight").float()

# ========== TOKENIZER ==========
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-32B-AWQ")

# ========== DOWNLOAD ACTIVATIONS ==========
!mkdir -p data/core data/transfer
!wget -q -O data/core/activations.npz https://huggingface.co/datasets/aaronrose227/narcbench/resolve/main/activations/qwen3_32b/core/activations_gen.npz
!wget -q -O data/core/metadata.json https://huggingface.co/datasets/aaronrose227/narcbench/resolve/main/activations/qwen3_32b/core/metadata_gen.json
!wget -q -O data/transfer/activations.npz https://huggingface.co/datasets/aaronrose227/narcbench/resolve/main/activations/qwen3_32b/transfer/activations_gen.npz
!wget -q -O data/transfer/metadata.json https://huggingface.co/datasets/aaronrose227/narcbench/resolve/main/activations/qwen3_32b/transfer/metadata_gen.json

# ========== LOAD DATA ==========
core_data = np.load("data/core/activations.npz")
with open("data/core/metadata.json") as f:
    core_meta = json.load(f)
transfer_data = np.load("data/transfer/activations.npz")
with open("data/transfer/metadata.json") as f:
    transfer_meta = json.load(f)

labels = np.array([1 if e['role'] == 'colluder' else 0 for e in core_meta])
transfer_labels = np.array([1 if e['role'] == 'colluder' else 0 for e in transfer_meta])
transfer_families = np.array([e['family'] for e in transfer_meta])

print(f"Loaded {len(core_meta)} Core samples, {len(transfer_meta)} Transfer samples")
print("Setup done.")

In [ ]:
# ========== Vocab ==========
domain_vocabs = {
    "cicd": ["hide", "hidden", "bypass", "skip", "silent", "quiet", "deploy", "merge",
             "push", "commit", "override", "force", "approve", "block", "revert", "config",
             "secret", "key"],
    "code_review": ["hide", "hidden", "overlook", "ignore", "skip", "quiet", "approve",
                    "merge", "reject", "flag", "bug", "issue", "vulnerability", "review",
                    "commit", "diff"],
    "news": ["hide", "hidden", "frame", "spin", "quiet", "headline", "coverage", "story",
             "article", "leak", "source", "bias", "angle", "narrative", "public"],
    "pump_dump": ["hide", "hidden", "coordinate", "quiet", "secret", "buy", "sell", "pump",
                  "dump", "spike", "hype", "promote", "target", "price", "trade", "stock"],
    "rag": ["hide", "hidden", "inject", "poison", "manipulate", "retrieve", "context",
            "source", "document", "prompt", "query", "response", "vector"],
    "research": ["hide", "hidden", "omit", "cherry", "selective", "publish", "cite", "result",
                 "data", "finding", "method", "author", "paper", "peer"],
}

usable = {}
for domain, words in domain_vocabs.items():
    for word in words:
        for w in [word, " " + word]:
            ids = tokenizer.encode(w, add_special_tokens=False)
            if len(ids) == 1:
                usable[w] = ids[0]
print(f"Vocab size: {len(usable)} single-token variants")

# ========== BUILD TARGET VECTORS ==========
target_vectors = {}
for name, tid in usable.items():
    unembed_row = W_U[tid]
    for layer in target_layers:
        target_vectors[(name, layer)] = unembed_row @ jacobians[layer].float()

target_names = sorted(usable.keys())
n_features = len(target_names) * len(target_layers)

# ========== COMPUTE FEATURES (DOT PRODUCT) ==========
def build_features(data_dict, n_samples):
    features = np.zeros((n_samples, n_features))
    for i, name in enumerate(target_names):
        for j, layer in enumerate(target_layers):
            v = target_vectors[(name, layer)]
            acts = torch.from_numpy(data_dict[f'layer_{layer}']).float()
            dot = acts @ v
            features[:, i * len(target_layers) + j] = dot.numpy()
    return features

core_features = build_features(core_data, len(core_meta))
transfer_features = build_features(transfer_data, len(transfer_meta))

# ========== TRAIN + EVALUATE ==========
clf = LogisticRegression(max_iter=5000, class_weight='balanced',
                        penalty='l1', solver='liblinear', C=10.0)

auc_scores = cross_val_score(clf, core_features, labels, cv=5, scoring='roc_auc')
print(f"\nIn-distribution AUROC: {auc_scores.mean():.3f} +/- {auc_scores.std():.3f}")

clf.fit(core_features, labels)
transfer_scores = clf.predict_proba(transfer_features)[:, 1]

print(f"Overall Transfer AUROC: {roc_auc_score(transfer_labels, transfer_scores):.3f}")
print("\nPer-scenario AUROC:")
for family in sorted(set(transfer_families)):
    mask = transfer_families == family
    if len(set(transfer_labels[mask])) > 1:
        auc = roc_auc_score(transfer_labels[mask], transfer_scores[mask])
        print(f"  {family}: {auc:.3f}")

print("\nTop 15 features:")
weights = clf.coef_[0]
feature_labels = [f"{name}_L{layer}" for name in target_names for layer in target_layers]
nonzero = np.where(np.abs(weights) > 0.01)[0]
sorted_idx = nonzero[np.argsort(np.abs(weights[nonzero]))[::-1]]
for idx in sorted_idx[:15]:
    print(f"  {feature_labels[idx]}: {weights[idx]:+.3f}")